# 06 — Arquitetura Spark

**EP · Data Science Knowledge Base | Big Data e Processamento Distribuído | Notebook 06/16**

**Execução:** use o kernel Python 3, reinicie o kernel e execute as células em ordem. Laboratórios Spark são opcionais e desativados por padrão.

**Abordagem:** os exemplos locais verificam conceitos e não representam um cluster distribuído.


## Materiais complementares

- [Guia Técnico](./reference_materials/06_spark_arquitetura.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_spark_arquitetura.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
Entender o ciclo de execução e criar sessões Spark sem alterar o sistema.

Ao final, você deve conseguir interpretar os resultados, justificar as escolhas e explicar suas limitações.

## Fundamentação teórica
Driver, executores, jobs, stages, tasks, lazy evaluation e particionamento.

**Atenção metodológica:** execução local facilita entendimento e depuração, mas não comprova ganhos de paralelismo em infraestrutura distribuída. Escalabilidade exige medir dados, nós, rede, memória, partições e operações de shuffle.

In [1]:
from pathlib import Path
import sys
DATASETS = Path('datasets')
assert DATASETS.is_dir(), 'Execute a partir da raiz da área 10 (pasta datasets não encontrada)'
print('Python:', sys.version.split()[0])
print('Pasta dos dados:', DATASETS.as_posix())

Python: 3.14.4
Pasta dos dados: datasets


## Demonstração 1 — construção e observação

In [2]:
import importlib.util
print('PySpark disponível:', importlib.util.find_spec('pyspark') is not None)
print('Java e Spark devem ser configurados em ambiente isolado; não execute apt-get dentro do notebook.')

PySpark disponível: False
Java e Spark devem ser configurados em ambiente isolado; não execute apt-get dentro do notebook.


## Interpretação
Observe o esquema, as unidades e os resultados. Antes de concluir que um algoritmo é eficiente, distinga **complexidade computacional**, **custo de comunicação** e **volume de dados**. Os cálculos locais ilustram a semântica; o Spark, quando disponível, ilustra a API distribuída.

## Demonstração 2 — validação e limitações

In [3]:
# Laboratório Spark real opcional, executado em interpretador isolado.
# Ative SOMENTE quando o ambiente PySpark e o Java já estiverem preparados.
import os, sys, subprocess, textwrap
if os.environ.get('RUN_SPARK_LABS') == '1':
    interpretador = os.environ.get('SPARK_LAB_PYTHON', sys.executable)
    assert os.path.isfile(interpretador), f'Interpretador não encontrado: {interpretador}'
    programa = textwrap.dedent(r'''from pyspark.sql import SparkSession
s=(SparkSession.builder.master('local[2]').appName('ep-area10-06')
   .config('spark.ui.enabled','false').config('spark.sql.shuffle.partitions','2').getOrCreate())
try:
    s.sparkContext.setLogLevel('ERROR')
    n=s.range(100).where('id % 3 = 0').count()
    assert n==34, n
    print('Spark local:',s.version,'| id divisíveis por 3:',n)
    print('SPARK_LAB_OK_06')
finally:s.stop()''')
    ambiente = os.environ.copy()
    ambiente.setdefault('SPARK_LOCAL_IP', '127.0.0.1')
    ambiente['PYSPARK_PYTHON'] = interpretador
    ambiente['PYSPARK_DRIVER_PYTHON'] = interpretador
    resultado = subprocess.run([interpretador, '-c', programa], cwd=os.getcwd(),
                               env=ambiente, text=True, capture_output=True, timeout=150)
    if resultado.stdout: print(resultado.stdout.strip())
    if resultado.returncode:
        print(resultado.stderr[-3500:])
        raise RuntimeError('O LABORATÓRIO SPARK 06 FALHOU; não registrar como aprovado.')
    assert 'SPARK_LAB_OK_06' in resultado.stdout, 'Marcador de sucesso Spark não encontrado'
else:
    print('SPARK NÃO EXECUTADO (opcional). Ative RUN_SPARK_LABS=1 e SPARK_LAB_PYTHON.')


Spark local: 4.1.4 | id divisíveis por 3: 34
SPARK_LAB_OK_06


## Boas práticas e limitações

- Execute com versões registradas, datasets identificados e caminhos relativos.
- Diferencie execução local, emulação lógica e infraestrutura distribuída real.
- Documente nulos, chaves, cardinalidade, custos e limitações das métricas.
- Não publique datasets ou resultados cuja redistribuição não esteja autorizada.
- Evite `collect()` de grandes tabelas e não confunda uma execução `local[*]` com cluster multinó.


## Exercícios
1. Explique que parte do exemplo poderia ser distribuída por chave ou partição e qual é o custo do shuffle.
2. Descreva uma validação de qualidade para os dados usados.
3. Indique um cenário em que o exemplo local deixaria de ser adequado.

## Respostas comentadas (orientativas)
1. Agregações por chave podem combinar resultados parciais, mas exigem redistribuir dados com a mesma chave; filtros e projeções usualmente evitam shuffle.
2. Verifique contagem, unicidade de chaves, percentual de valores ausentes, tipos, intervalos e consistência das unidades.
3. Quando dados, latência, concorrência ou exigências de falha excederem a capacidade de uma máquina; nesse caso, considere Spark/Hadoop e meça os custos adicionais.

## Síntese

O capítulo 06 reuniu conceitos e verificações de **sessões, driver e execução local**. Os exemplos de pequena escala não comprovam desempenho em cluster. Consulte apenas referências complementares efetivamente publicadas e documentadas.


## Revisão científica — auditoria e aplicação

**Ponto de atenção deste capítulo:** driver e executor. Esta seção complementa os exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **driver e executor** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — Laboratório PySpark opcional: sessão local

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.

In [4]:
# A sessão Spark real está consolidada no laboratório opcional acima.
# Este espaço mantém a posição histórica do segundo exemplo, sem abrir nova sessão.
print('Experimento Spark único por notebook; confira a mensagem de sucesso quando ativado.')


Experimento Spark único por notebook; confira a mensagem de sucesso quando ativado.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **driver e executor** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
